<a href="https://colab.research.google.com/github/Psynotch/firstrep/blob/master/Session12_Height_10000_Verification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip -q install python-flint==0.9.0 mpmath==1.3.0

In [2]:
import csv
import json
import time
import hashlib
import platform
import importlib.metadata

import mpmath as mp

from fractions import Fraction
from flint import acb, arb, arb_series, ctx

ctx.prec = 256
mp.mp.dps = 80

T_LO = "10"
T_MID = "5000"
T_HI = "10000"

print("SESSION 12 INITIALIZED")
print("Python:", platform.python_version())
print("python-flint:", importlib.metadata.version("python-flint"))
print("mpmath:", importlib.metadata.version("mpmath"))
print("Precision:", ctx.prec, "bits")

SESSION 12 INITIALIZED
Python: 3.13.18
python-flint: 0.9.0
mpmath: 1.3.0
Precision: 256 bits


In [3]:
print("SESSION 12: TURING ZERO COUNTS")
print("--------------------------------")

start = time.time()

N10_ball = arb(T_LO).zeta_nzeros()
N5000_ball = arb(T_MID).zeta_nzeros()
N10000_ball = arb(T_HI).zeta_nzeros()

N10 = N10_ball.unique_fmpz()
N5000 = N5000_ball.unique_fmpz()
N10000 = N10000_ball.unique_fmpz()

if any(x is None for x in [N10, N5000, N10000]):
    raise RuntimeError("A count did not isolate a unique integer.")

N10 = int(N10)
N5000 = int(N5000)
N10000 = int(N10000)

count_10_5000 = N5000 - N10
count_5000_10000 = N10000 - N5000
total_count = N10000 - N10

print("N(10):", N10)
print("N(5000):", N5000)
print("N(10000):", N10000)

print("\nPreviously verified range:", count_10_5000)
print("New range (5000–10000):", count_5000_10000)
print("TOTAL ZEROS (10–10000):", total_count)

print("\nPrevious result matches:",
      count_10_5000 == 4520)

print("Elapsed seconds:", round(time.time() - start, 3))

assert count_10_5000 == 4520
assert count_5000_10000 > 0

SESSION 12: TURING ZERO COUNTS
--------------------------------
N(10): 0
N(5000): 4520
N(10000): 10142

Previously verified range: 4520
New range (5000–10000): 5622
TOTAL ZEROS (10–10000): 10142

Previous result matches: True
Elapsed seconds: 0.004


In [4]:
def compute_zero_set(bits):

    ctx.prec = bits

    start = time.time()

    lower_count = arb(T_LO).zeta_nzeros().unique_fmpz()
    upper_count = arb(T_HI).zeta_nzeros().unique_fmpz()

    if lower_count is None or upper_count is None:
        raise RuntimeError(f"Count failed at {bits} bits.")

    lower_count = int(lower_count)
    upper_count = int(upper_count)

    count = upper_count - lower_count

    zero_list = list(
        acb.zeta_zeros(lower_count + 1, count)
    )

    if len(zero_list) != count:
        raise RuntimeError("Unexpected number of zero enclosures.")

    return {
        "bits": bits,
        "count": count,
        "zeros": zero_list,
        "seconds": time.time() - start
    }


result_192 = compute_zero_set(192)
result_256 = compute_zero_set(256)

for result in [result_192, result_256]:

    print("\nPrecision:", result["bits"], "bits")
    print("Zero enclosures:", result["count"])
    print("Execution time:", round(result["seconds"], 2), "seconds")

assert result_192["count"] == total_count
assert result_256["count"] == total_count

print("\nFIRST THREE ENCLOSURES")

for z in result_256["zeros"][:3]:
    print(z)

print("\nLAST THREE ENCLOSURES")

for z in result_256["zeros"][-3:]:
    print(z)


Precision: 192 bits
Zero enclosures: 10142
Execution time: 228.69 seconds

Precision: 256 bits
Zero enclosures: 10142
Execution time: 255.33 seconds

FIRST THREE ENCLOSURES
0.5000000000000000000000000000000000000000000000000000000000000000000000000000 + [14.13472514173469379045725198356247027078425711569924317568556746014996342981 +/- 8.84e-76]j
0.5000000000000000000000000000000000000000000000000000000000000000000000000000 + [21.02203963877155499262847959389690277733434052490278175462952040358759858607 +/- 1.61e-75]j
0.5000000000000000000000000000000000000000000000000000000000000000000000000000 + [25.01085758014568876321379099256282181865954967255799667249654200674509209844 +/- 1.80e-75]j

LAST THREE ENCLOSURES
0.5000000000000000000000000000000000000000000000000000000000000000000000000000 + [9996.556933962111869093494491195931888891193469171869077911073342583280469791 +/- 4.56e-73]j
0.5000000000000000000000000000000000000000000000000000000000000000000000000000 + [9998.0948603349750323

In [5]:
#Cross-precision comparison
print("CROSS-PRECISION COMPARISON")
print("--------------------------")

overlap_count = 0
nonoverlapping = []

for i, (z192, z256) in enumerate(
    zip(result_192["zeros"], result_256["zeros"]),
    start=1
):

    if z192.overlaps(z256):
        overlap_count += 1
    else:
        nonoverlapping.append(i)

print("Total compared:", total_count)
print("Overlapping:", overlap_count)
print("Non-overlapping:", len(nonoverlapping))

if nonoverlapping:
    print("Problematic indices:", nonoverlapping[:20])

assert overlap_count == total_count

CROSS-PRECISION COMPARISON
--------------------------
Total compared: 10142
Overlapping: 10142
Non-overlapping: 0


In [7]:
#Generate adaptive intervals
ctx.prec = 256

def rational_ball(q):
    return arb(f"{q.numerator}/{q.denominator}")


zeros = result_256["zeros"]

centers = [
    Fraction(z.imag.mid().str(70, radius=False))
    for z in zeros
]

records = []

for i, zero in enumerate(zeros):

    center = centers[i]
    gamma = zero.imag

    radius = Fraction(1, 100)

    if i > 0:
        radius = min(
            radius,
            (center - centers[i - 1]) / 4
        )

    if i + 1 < len(centers):
        radius = min(
            radius,
            (centers[i + 1] - center) / 4
        )

    radius = min(
        radius,
        (center - Fraction(10)) / 4,
        (Fraction(10000) - center) / 4
    )

    if radius <= 0:
        raise RuntimeError(f"Invalid interval at zero {i+1}")

    left = center - radius
    right = center + radius

    contained = (
        rational_ball(left).upper() < gamma.lower()
        and gamma.upper() < rational_ball(right).lower()
    )

    if not contained:
        raise RuntimeError(
            f"Zero {i+1}: enclosure not contained in interval."
        )

    records.append({
        "index": i + 1,
        "zero": zero,
        "gamma": gamma,
        "left": left,
        "right": right,
        "radius": radius
    })


for i in range(len(records) - 1):

    if not records[i]["right"] < records[i + 1]["left"]:
        raise RuntimeError(f"Intervals {i+1} and {i+2} overlap.")


print("Intervals generated:", len(records))
print("All enclosures contained: True")
print("All intervals disjoint: True")
print("All intervals inside (10,10000): True")

print("\nFirst zero:", records[0]["gamma"])
print("Last zero:", records[-1]["gamma"])

Intervals generated: 10142
All enclosures contained: True
All intervals disjoint: True
All intervals inside (10,10000): True

First zero: [14.13472514173469379045725198356247027078425711569924317568556746014996342981 +/- 8.84e-76]
Last zero: [9998.850397089674049057631756625830373404247354886196460449629064822885379673 +/- 6.24e-73]


In [8]:
#Hardy Z-function verification
def hardy_z(t_fraction):

    t = rational_ball(t_fraction)

    series = arb_series(
        [t, 1],
        prec=2
    ).riemann_siegel_z()

    return series.coeffs()[0]


def certified_sign(value):

    sign = value.sgn()

    if sign == arb(1):
        return "+"

    if sign == arb(-1):
        return "-"

    return "UNKNOWN"


verification_rows = []
failed = []

print("SESSION 12: HARDY Z VERIFICATION")
print("--------------------------------")

start = time.time()

for i, record in enumerate(records, start=1):

    z_left = hardy_z(record["left"])
    z_right = hardy_z(record["right"])

    sign_left = certified_sign(z_left)
    sign_right = certified_sign(z_right)

    passed = (
        sign_left != "UNKNOWN"
        and sign_right != "UNKNOWN"
        and sign_left != sign_right
    )

    if not passed:
        failed.append(record["index"])

    verification_rows.append({
        "zero_index": record["index"],
        "left": str(record["left"]),
        "right": str(record["right"]),
        "left_sign": sign_left,
        "right_sign": sign_right,
        "passed": passed
    })

    if i % 500 == 0 or i == len(records):
        print(
            f"Checked {i}/{len(records)} | "
            f"Failed: {len(failed)}"
        )

verified = len(records) - len(failed)

print("\nVERIFIED:", verified)
print("FAILED:", len(failed))
print("Failed indices:", failed)

print("Elapsed seconds:", round(time.time() - start, 2))

assert verified == total_count

SESSION 12: HARDY Z VERIFICATION
--------------------------------
Checked 500/10142 | Failed: 0
Checked 1000/10142 | Failed: 0
Checked 1500/10142 | Failed: 0
Checked 2000/10142 | Failed: 0
Checked 2500/10142 | Failed: 0
Checked 3000/10142 | Failed: 0
Checked 3500/10142 | Failed: 0
Checked 4000/10142 | Failed: 0
Checked 4500/10142 | Failed: 0
Checked 5000/10142 | Failed: 0
Checked 5500/10142 | Failed: 0
Checked 6000/10142 | Failed: 0
Checked 6500/10142 | Failed: 0
Checked 7000/10142 | Failed: 0
Checked 7500/10142 | Failed: 0
Checked 8000/10142 | Failed: 0
Checked 8500/10142 | Failed: 0
Checked 9000/10142 | Failed: 0
Checked 9500/10142 | Failed: 0
Checked 10000/10142 | Failed: 0
Checked 10142/10142 | Failed: 0

VERIFIED: 10142
FAILED: 0
Failed indices: []
Elapsed seconds: 85.62


In [9]:
#Independent mpmath cross-check
mp.mp.dps = 80

sample_positions = sorted(set([
    1, 2, 3, 10,
    total_count // 20,
    total_count // 10,
    total_count // 5,
    total_count // 4,
    total_count // 3,
    total_count // 2,
    2 * total_count // 3,
    3 * total_count // 4,
    4 * total_count // 5,
    9 * total_count // 10,
    19 * total_count // 20,
    total_count - 2,
    total_count - 1,
    total_count
]))


def mpmath_sign(t_fraction):

    t = (
        mp.mpf(t_fraction.numerator)
        / t_fraction.denominator
    )

    value = mp.siegelz(t)

    if value > 0:
        return "+"

    if value < 0:
        return "-"

    return "UNKNOWN"


mpmath_checks = []

print("MPMATH CROSS-CHECK")
print("------------------")

for position in sample_positions:

    record = records[position - 1]
    arb_row = verification_rows[position - 1]

    ml = mpmath_sign(record["left"])
    mr = mpmath_sign(record["right"])

    agrees = (
        ml == arb_row["left_sign"]
        and mr == arb_row["right_sign"]
    )

    mpmath_checks.append({
        "position": position,
        "mpmath_left": ml,
        "arb_left": arb_row["left_sign"],
        "mpmath_right": mr,
        "arb_right": arb_row["right_sign"],
        "agrees": agrees
    })

    print(
        f"Zero {position}: "
        f"left={ml}/{arb_row['left_sign']}, "
        f"right={mr}/{arb_row['right_sign']}, "
        f"AGREE={agrees}"
    )


mpmath_agreements = sum(
    item["agrees"] for item in mpmath_checks
)

print("\nAgreements:", mpmath_agreements, "/", len(sample_positions))

MPMATH CROSS-CHECK
------------------
Zero 1: left=-/-, right=+/+, AGREE=True
Zero 2: left=+/+, right=-/-, AGREE=True
Zero 3: left=-/-, right=+/+, AGREE=True
Zero 10: left=+/+, right=-/-, AGREE=True
Zero 507: left=-/-, right=+/+, AGREE=True
Zero 1014: left=+/+, right=-/-, AGREE=True
Zero 2028: left=+/+, right=-/-, AGREE=True
Zero 2535: left=-/-, right=+/+, AGREE=True
Zero 3380: left=+/+, right=-/-, AGREE=True
Zero 5071: left=-/-, right=+/+, AGREE=True
Zero 6761: left=-/-, right=+/+, AGREE=True
Zero 7606: left=+/+, right=-/-, AGREE=True
Zero 8113: left=-/-, right=+/+, AGREE=True
Zero 9127: left=-/-, right=+/+, AGREE=True
Zero 9634: left=+/+, right=-/-, AGREE=True
Zero 10140: left=+/+, right=-/-, AGREE=True
Zero 10141: left=-/-, right=+/+, AGREE=True
Zero 10142: left=+/+, right=-/-, AGREE=True

Agreements: 18 / 18


In [10]:
with open(
    "Session12_full_zero_certificate.csv",
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "zero_index",
            "complex_zero_enclosure",
            "ordinate_enclosure",
            "left",
            "right",
            "left_sign",
            "right_sign",
            "passed"
        ]
    )

    writer.writeheader()

    for record, check in zip(records, verification_rows):

        writer.writerow({
            "zero_index": record["index"],
            "complex_zero_enclosure": str(record["zero"]),
            "ordinate_enclosure": str(record["gamma"]),
            "left": str(record["left"]),
            "right": str(record["right"]),
            "left_sign": check["left_sign"],
            "right_sign": check["right_sign"],
            "passed": check["passed"]
        })


csv_path = "Session12_full_zero_certificate.csv"

with open(csv_path, "rb") as f:
    csv_hash = hashlib.sha256(f.read()).hexdigest()

with open(csv_path, newline="", encoding="utf-8") as f:
    saved_rows = list(csv.DictReader(f))

csv_valid = (
    len(saved_rows) == total_count
    and all(row["passed"] == "True" for row in saved_rows)
)

summary = {
    "session": 12,
    "region": [0, 1, 10, 10000],
    "precisions_bits": [192, 256],
    "N_10": N10,
    "N_5000": N5000,
    "N_10000": N10000,
    "zeros_10_to_5000": count_10_5000,
    "zeros_5000_to_10000": count_5000_10000,
    "total_zeros": total_count,
    "cross_precision_overlaps": overlap_count,
    "hardy_sign_checks": verified,
    "failed_intervals": failed,
    "mpmath_sample_checks": len(sample_positions),
    "mpmath_agreements": mpmath_agreements,
    "csv_rows": len(saved_rows),
    "csv_valid": csv_valid,
    "csv_sha256": csv_hash
}

with open(
    "Session12_verification_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, indent=2)

print("CSV SHA-256:", csv_hash)
print("\nSESSION 12 SUMMARY")
print("==================")
print(json.dumps(summary, indent=2))

CSV SHA-256: 39f9e8f6fe58f93b4c3ade2c7efc40cd49c6fe91662be75004783b63b864a417

SESSION 12 SUMMARY
{
  "session": 12,
  "region": [
    0,
    1,
    10,
    10000
  ],
  "precisions_bits": [
    192,
    256
  ],
  "N_10": 0,
  "N_5000": 4520,
  "N_10000": 10142,
  "zeros_10_to_5000": 4520,
  "zeros_5000_to_10000": 5622,
  "total_zeros": 10142,
  "cross_precision_overlaps": 10142,
  "hardy_sign_checks": 10142,
  "failed_intervals": [],
  "mpmath_sample_checks": 18,
  "mpmath_agreements": 18,
  "csv_rows": 10142,
  "csv_valid": true,
  "csv_sha256": "39f9e8f6fe58f93b4c3ade2c7efc40cd49c6fe91662be75004783b63b864a417"
}


In [11]:
final_success = (
    total_count == len(zeros)
    and result_192["count"] == total_count
    and result_256["count"] == total_count
    and overlap_count == total_count
    and verified == total_count
    and len(failed) == 0
    and mpmath_agreements == len(sample_positions)
    and csv_valid
)

print("SESSION 12 FINAL VERIFICATION")
print("=============================")

print("Total zeros:", total_count)
print("Cross-precision matches:", overlap_count)
print("Hardy Z verification:", verified)
print("Failed intervals:", len(failed))
print("mpmath agreements:", mpmath_agreements)
print("Certificate file valid:", csv_valid)

if final_success:
    print("\nSESSION 12: SUCCESS")
else:
    print("\nSESSION 12: NOT YET COMPLETE")

SESSION 12 FINAL VERIFICATION
Total zeros: 10142
Cross-precision matches: 10142
Hardy Z verification: 10142
Failed intervals: 0
mpmath agreements: 18
Certificate file valid: True

SESSION 12: SUCCESS
